In [ ]:
# =====================================================
# 1. CARREGAMENTO DA MOBILENETV2 (substitui a ResNet50)
# =====================================================
from tensorflow.keras.applications import MobileNetV2

modelo_base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
print("✅ Modelo MobileNetV2 carregado com sucesso!")

✅ Modelo MobileNetV2 carregado com sucesso!


In [ ]:
# =====================================================
# 2. CRIAÇÃO DO MODELO (CLASSIFICAÇÃO BINÁRIA)
# =====================================================
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

# Congela o modelo base
modelo_base.trainable = False

# Modelo com 1 neurônio de saída (sigmoid)
modelo = models.Sequential([
    modelo_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),       # menos neurônios para evitar overfitting
    layers.Dropout(0.5),                       # dropout mais alto
    layers.Dense(1, activation='sigmoid')      # saída binária
])

# Compilação para binário
modelo.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(modelo.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 82,049 (320.50 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

None


In [ ]:
# =====================================================
# 3. MONTAR O DRIVE E VERIFICAR CAMINHOS (evita FileNotFoundError)
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

import os
caminho_base = '/content/drive/MyDrive/dataset_shimeji'

# Verifica se o diretório base existe
if not os.path.exists(caminho_base):
    raise FileNotFoundError(f"❌ Diretório base não encontrado: {caminho_base}")
else:
    print(f"✅ Diretório base encontrado: {caminho_base}")
    print("📁 Conteúdo do diretório base:")
    for item in os.listdir(caminho_base):
        print(f"   - {item}")

# Verifica se as pastas de treino e validação existem
treino_path = os.path.join(caminho_base, 'treino')
validacao_path = os.path.join(caminho_base, 'validacao')

if not os.path.exists(treino_path):
    raise FileNotFoundError(f"❌ Pasta de treino não encontrada: {treino_path}")
if not os.path.exists(validacao_path):
    raise FileNotFoundError(f"❌ Pasta de validação não encontrada: {validacao_path}")

print("✅ Estrutura de pastas verificada com sucesso.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Diretório base encontrado: /content/drive/MyDrive/dataset_shimeji
📁 Conteúdo do diretório base:
   - treino
   - validacao
   - teste
   - modelo_shimeji.keras
   - modelo_mobilenet_shimeji.keras
   - modelo_mobilenet_shimeji_2.keras
✅ Estrutura de pastas verificada com sucesso.


In [ ]:
# =====================================================
# 4. GERADORES DE DADOS (BINÁRIO)
# =====================================================
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2
import numpy as np

# Função para aplicar desfoque aleatório
def aplicar_desfoque(img):
    if np.random.rand() < 0.3:
        ksize = np.random.choice([3, 5, 7])
        img = cv2.GaussianBlur(img, (ksize, ksize), 0)
    return img

# Função para adicionar ruído
def adicionar_ruido(img):
    if np.random.rand() < 0.2:
        noise = np.random.normal(0, 0.05, img.shape)
        img = img + noise
        img = np.clip(img, 0, 1)
    return img

def preprocessamento_personalizado(img):
    img = aplicar_desfoque(img)
    img = adicionar_ruido(img)
    return img

# Aumento de dados (agressivo)
treino_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.4,
    height_shift_range=0.4,
    shear_range=0.3,
    zoom_range=0.4,
    horizontal_flip=True,
    brightness_range=[0.5, 1.5],
    preprocessing_function=preprocessamento_personalizado,
    fill_mode='nearest'
)

validacao_datagen = ImageDataGenerator(rescale=1./255)

# Caminhos (certifique-se de que as pastas 'anomalia' existem)
treino_generator = treino_datagen.flow_from_directory(
    treino_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'          # <-- MUDANÇA: binary
)

validacao_generator = validacao_datagen.flow_from_directory(
    validacao_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'          # <-- MUDANÇA: binary
)

print(f"✅ Treino: {treino_generator.samples} imagens")
print(f"✅ Validação: {validacao_generator.samples} imagens")

Found 240 images belonging to 2 classes.
Found 60 images belonging to 2 classes.
✅ Treino: 240 imagens
✅ Validação: 60 imagens


In [ ]:
# =====================================================
# 5. TREINO DO MODELO (BINÁRIO COM CLASS WEIGHT)
# =====================================================
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# --- 5a. Calcular os pesos das classes ---
# O flow_from_directory com class_mode='binary' mapeia as classes:
# classe 0 = primeira pasta em ordem alfabética (anomalia)
# classe 1 = segunda pasta (saudavel)
# Como as pastas são ['anomalia', 'saudavel'], temos:
# classe 0 = anomalia (160 imagens)
# classe 1 = saudavel (80 imagens)

classes = treino_generator.classes  # array com 0 e 1
pesos = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes),  # [0, 1]
    y=classes
)
class_weight = dict(enumerate(pesos))

print(f"⚖️ Pesos das classes:")
print(f"   Classe 0 (anomalia): {class_weight[0]:.4f}")
print(f"   Classe 1 (saudável): {class_weight[1]:.4f}")

# --- 5b. Callbacks ---
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    'melhor_modelo_binario.keras',
    save_best_only=True,
    monitor='val_accuracy'
)

# --- 5c. Treino ---
historico = modelo.fit(
    treino_generator,
    epochs=50,
    validation_data=validacao_generator,
    class_weight=class_weight,  # <-- ADICIONADO: compensa o desbalanceamento
    callbacks=[early_stop, checkpoint],
    verbose=1
)

print("✅ Treino concluído!")

⚖️ Pesos das classes:
   Classe 0 (anomalia): 0.7500
   Classe 1 (saudável): 1.5000
Epoch 1/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 114s 13s/step - accuracy: 0.6292 - loss: 0.8683 - val_accuracy: 0.6667 - val_loss: 0.6052
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 27s 3s/step - accuracy: 0.5583 - loss: 0.7745 - val_accuracy: 0.5000 - val_loss: 0.6380
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.5958 - loss: 0.6790 - val_accuracy: 0.3500 - val_loss: 0.6496
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 26s 3s/step - accuracy: 0.6292 - loss: 0.6240 - val_accuracy: 0.3500 - val_loss: 0.6581
Epoch 5/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 25s 3s/step - accuracy: 0.6417 - loss: 0.6532 - val_accuracy: 0.4333 - val_loss: 0.6638
Epoch 6/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.7167 - loss: 0.5720 - val_accuracy: 0.4833 - val_loss: 0.6712
Epoch 7/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 24s 3s/step - accuracy: 0.6958 - loss: 0.5760 - val_accuracy: 0.5000 - val_loss: 0.6839
Epoch 8/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 26s 3s/ste

In [ ]:
# =====================================================
# 5.1 TESTE - TAXA DE FALSOS POSITIVOS (FPR) e FALSOS NEGATIVOS (FNR)
# =====================================================
from sklearn.metrics import confusion_matrix
import numpy as np

# Previsões no conjunto de validação
val_preds = modelo.predict(validacao_generator, verbose=0)
val_labels = validacao_generator.classes  # 0 = anomalia, 1 = saudável

# Transforma as predições em classes binárias (threshold 0.5)
val_pred_bin = (val_preds > 0.5).astype(int).flatten()

# Matriz de confusão
tn, fp, fn, tp = confusion_matrix(val_labels, val_pred_bin).ravel()

# Cálculo das métricas
total = len(val_labels)
acuracia = (tp + tn) / total
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
especificidade = tn / (tn + fp) if (tn + fp) > 0 else 0
sensibilidade = tp / (tp + fn) if (tp + fn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

print("="*50)
print("📊 RELATÓRIO DE VALIDAÇÃO (BINÁRIO)")
print("="*50)
print(f"Total de imagens na validação: {total}")
print(f"  - Anomalias (reais): {np.sum(val_labels == 0)}")
print(f"  - Saudáveis (reais): {np.sum(val_labels == 1)}")
print("-"*50)
print(f"✅ Acurácia: {acuracia:.2%} (meta: ≥ 50%)")
print(f"✅ Sensibilidade (Recall da anomalia): {sensibilidade:.2%}")
print(f"⚠️  Especificidade (Taxa de acerto em saudáveis): {especificidade:.2%}")
print(f"🎯 Falso Positivo (FPR): {fpr:.2%} (meta: < 10%)")
print(f"❌ FNR (Falsos Negativos): {fnr:.2%}")
print("-"*50)
print(f"Matriz de confusão:")
print(f"               Predito")
print(f"              Anomalia  Saudável")
print(f"Anomalia (real)    {tp}        {fn}")
print(f"Saudável (real)    {fp}        {tn}")
print("="*50)

if fpr <= 0.10:
    print("🎉✅ FPR ATENDE AO CRITÉRIO (< 10%)!")
else:
    print(f"❌ FPR está em {fpr:.2%}, ACIMA da meta de 10%.")

📊 RELATÓRIO DE VALIDAÇÃO (BINÁRIO)
Total de imagens na validação: 60
  - Anomalias (reais): 40
  - Saudáveis (reais): 20
--------------------------------------------------
✅ Acurácia: 68.33% (meta: ≥ 50%)
✅ Sensibilidade (Recall da anomalia): 10.00%
⚠️  Especificidade (Taxa de acerto em saudáveis): 97.50%
🎯 Falso Positivo (FPR): 2.50% (meta: < 10%)
FNR (Falsos Negativos): 90.00%
--------------------------------------------------
Matriz de confusão:
               Predito
              Anomalia  Saudável
Anomalia (real)    2        18
Saudável (real)    1        39
🎉✅ FPR ATENDE AO CRITÉRIO (< 10%)!


In [ ]:
# =====================================================
# 6. GRÁFICOS DE ACURÁCIA E PERDA
# =====================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(historico.history['accuracy'], label='Treino')
plt.plot(historico.history['val_accuracy'], label='Validação')
plt.title('Acurácia do Modelo (MobileNetV2)')
plt.xlabel('Épocas')
plt.ylabel('Acurácia')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(historico.history['loss'], label='Treino')
plt.plot(historico.history['val_loss'], label='Validação')
plt.title('Perda do Modelo (MobileNetV2)')
plt.xlabel('Épocas')
plt.ylabel('Perda')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# 7. SALVAR O MODELO NO GOOGLE DRIVE
# =====================================================
modelo.save('/content/drive/MyDrive/dataset_shimeji/modelo_mobilenet_shimeji.keras')
print("✅ Modelo salvo no Google Drive como 'modelo_mobilenet_shimeji.keras'")

In [ ]:
# =====================================================
# 8. DOWNLOAD DO MELHOR MODELO (checkpoint)
# =====================================================
from google.colab import files

if os.path.exists('melhor_modelo_mobilenet.keras'):
    files.download('melhor_modelo_mobilenet.keras')
    print("✅ Download do melhor modelo iniciado!")
else:
    print("❌ Arquivo 'melhor_modelo_mobilenet.keras' não encontrado.")
    print("📂 Arquivos disponíveis no diretório atual:")
    for arquivo in os.listdir('.'):
        print(f"   - {arquivo}")

In [ ]:
# =====================================================
# 9. FUNÇÃO PARA TESTAR UMA IMAGEM
# =====================================================
from tensorflow.keras.preprocessing import image
import numpy as np

def classificar_cogumelo(caminho_da_foto):
    """
    Classifica uma imagem de cogumelo usando o modelo treinado (MobileNetV2).

    Args:
        caminho_da_foto (str): Caminho completo para a imagem no Drive.

    Returns:
        tuple: (classe_prevista, confianca)
    """
    # Carrega e prepara a imagem
    img = image.load_img(caminho_da_foto, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0  # Normaliza igual foi feito no treino

    # Faz a previsão
    previsao = modelo.predict(img_array, verbose=0)

    # Ordem alfabética das classes (como definido no flow_from_directory)
    classes = ['com_fungo', 'defeituoso', 'saudavel']

    classe_prevista = classes[np.argmax(previsao)]
    confianca = np.max(previsao) * 100

    # Exibe o resultado detalhado
    print(f"\n{'='*50}")
    print(f"📸 Imagem: {caminho_da_foto.split('/')[-1]}")
    print(f"{'='*50}")
    print(f"🔍 Resultado: {classe_prevista.upper()}")
    print(f"📊 Confiança: {confianca:.2f}%")
    print(f"\n📈 Probabilidades por classe:")
    for i, classe in enumerate(classes):
        barra = '█' * int(previsao[0][i] * 50)
        print(f"   {classe:12s}: {previsao[0][i]*100:5.2f}% {barra}")
    print(f"{'='*50}\n")

    return classe_prevista, confianca

In [ ]:
# =====================================================
# 10. EXEMPLOS DE USO DA FUNÇÃO DE TESTE
# =====================================================
# Exemplo: testar uma imagem da pasta de validação
# classificar_cogumelo('/content/drive/MyDrive/dataset_shimeji/validacao/saudavel/nome_da_foto.jpg')

# Exemplo: testar uma imagem da pasta de treino
# classificar_cogumelo('/content/drive/MyDrive/dataset_shimeji/treino/com_fungo/nome_da_foto.jpg')